# Regional ERA5 ingestion from Copernicus CDS

Edit the global settings below, then run the cells in order. Start with one UTC day. Set `RUN_DOWNLOAD = True` when ready.

Install `.[weather,notebooks]` into the notebook kernel environment. A CDS account must accept the [ERA5 single-level dataset terms](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download). Configure `CDSAPI_KEY` securely through environment settings or `~/.cdsapirc`; never paste the token into this notebook.

Each UTC-day Parquet file is shared by every region. Pulling Alaska later extends the same files; overlapping observations update by timestamp, location, and variable.

In [ ]:
# GLOBAL SETTINGS — edit this cell before running.
from datetime import date

REGION_NAME = "California"
NORTH = 42.0
WEST = -124.5
SOUTH = 32.5
EAST = -114.0

START_DATE = date(2000, 1, 1)  # Inclusive UTC calendar day.
END_DATE = date(2000, 1, 2)    # Exclusive UTC calendar day.
VARIABLES = ("2m_temperature",)
WORKERS = 2
REFRESH = False               # True re-fetches and updates matching observations.
KEEP_RAW = False              # Successful NetCDF responses are otherwise removed.
LAKE_ROOT = None              # None -> <project>/data/lake/weather/era5
RAW_ROOT = None               # None -> <project>/data/raw/weather/era5
RUN_DOWNLOAD = False          # Set True after checking region, dates, and credentials.

## 1. Load the project and check credential presence

In [ ]:
import logging
import os
import sys
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/comm_research").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the commodity-research checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import polars as pl
from comm_research.markets.weather.models.grid import WeatherGrid
from comm_research.markets.weather.scrapers.era5 import (
    ERA5Scraper, DEFAULT_LAKE_ROOT, DEFAULT_RAW_ROOT, KEYS,
)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

credential_file = Path(os.environ.get("CDSAPI_RC", str(Path.home() / ".cdsapirc"))).expanduser()
print("CDS credential source present:", bool(os.environ.get("CDSAPI_KEY")) or credential_file.is_file())
print("Only presence is checked here; CDS validates access when a download is requested.")

## 2. Define the weather grid and inspect request size

In [ ]:
grid = WeatherGrid(north=NORTH, west=WEST, south=SOUTH, east=EAST, name=REGION_NAME)
lake_root = Path(LAKE_ROOT).expanduser() if LAKE_ROOT is not None else DEFAULT_LAKE_ROOT
raw_root = Path(RAW_ROOT).expanduser() if RAW_ROOT is not None else DEFAULT_RAW_ROOT
print(grid)
print("Aligned CDS areas [north, west, south, east]:", grid.cds_areas)
print("Grid points:", f"{grid.point_count:,}")
print("Expected observations per complete day:", f"{grid.point_count * 24 * len(VARIABLES):,}")
print("UTC date range:", START_DATE, "through", END_DATE, "(exclusive)")
print("Lake:", lake_root)

## 3. Pull data and merge it into daily Parquet partitions

In [ ]:
summary = None
if RUN_DOWNLOAD:
    scraper = ERA5Scraper(
        grid, variables=VARIABLES, lake_root=lake_root, raw_root=raw_root,
        workers=WORKERS, keep_raw=KEEP_RAW,
    )
    summary = scraper.run(START_DATE, END_DATE, refresh=REFRESH)
    print(summary)
else:
    print("Download disabled. Set RUN_DOWNLOAD = True in the global settings to start.")

## 4. Inspect the requested region in the shared lake

Temperature is stored in kelvin. `value - 273.15` converts temperature to Celsius. Other variables retain the units reported by CDS. Radiation and precipitation retain their source accumulation semantics.

To add Alaska, change the bounds and rerun. For an antimeridian-crossing box, use `WEST > EAST`, e.g. `WEST = 170`, `EAST = -130`. The scraper splits CDS requests and stores signed longitudes in `[-180, 180)`.

In [ ]:
if lake_root.exists() and any(lake_root.rglob("*.parquet")):
    selected = (
        pl.scan_parquet(str(lake_root / "**/*.parquet"), hive_partitioning=True)
        .filter(
            pl.col("latitude").is_in(grid.latitudes)
            & pl.col("longitude").is_in(grid.longitudes)
            & pl.col("variable").is_in(VARIABLES)
            & pl.col("timestamp").dt.date().is_between(START_DATE, END_DATE, closed="left")
        )
    )
    display(selected.sort(KEYS).head(10).collect())
    display(selected.group_by("variable", "unit").agg(
        pl.len().alias("observations"),
        pl.col("value").min().alias("minimum"),
        pl.col("value").max().alias("maximum"),
    ).collect())
else:
    print("No ERA5 Parquet files yet. Run the download cell after configuring CDS access.")